# 02. Daikibo Forensic HR Audit & Gender Pay Equity Modeling
**Objective:** Audit internal corporate compensation records across Daikibo's manufacturing plants and organizational seniority tiers, apply multi-condition logical governance modeling, and identify exposure to gender pay disparities.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style="whitegrid")


## 1. Load Baseline Equality Data
Ingest the compensation table from `../data/raw/Task 5 Equality Table.xlsx`.


In [ ]:
raw_path = '../data/raw/Task 5 Equality Table.xlsx'

df_eq = pd.read_excel(raw_path)
print(f"Total roles audited: {len(df_eq)}")
df_eq.head(10)


## 2. Implement Forensic Classification Logic
We apply the rule provided by the Forensic Technology lead:
* **Fair:** Score within $\pm 10$ ($[-10, +10]$)
* **Unfair:** Score between $\pm 11$ and $\pm 20$ ($[-20, -11] \cup [11, 20]$)
* **Highly Discriminative:** Score strictly less than $-20$ or greater than $+20$ ($<-20$ or $>+20$)


In [ ]:
def classify_equality(score):
    if abs(score) <= 10:
        return "Fair"
    elif abs(score) <= 20:
        return "Unfair"
    else:
        return "Highly Discriminative"

# Populate classification column
df_eq['Equality class'] = df_eq['Equality Score'].apply(classify_equality)

print("Classification Breakdown Across Entire Organization:")
print(df_eq['Equality class'].value_counts())


## 3. Disparity Distribution by Hierarchical Seniority Tier
Examine whether gender pay gaps correlate with organizational seniority.


In [ ]:
role_order = [
    'C-Level', 'VP', 'Director', 'Sr. Manager', 'Manager', 'Jr. Manager',
    'Operational Support', 'Machine Operator', 'Engineer', 'Sr. Engineer', 'Jr. Engineer'
]

role_analysis = df_eq.groupby('Job Role').agg(
    role_count=('Equality Score', 'count'),
    mean_score=('Equality Score', 'mean'),
    min_score=('Equality Score', 'min'),
    max_score=('Equality Score', 'max')
).reindex(role_order).reset_index()

role_analysis


## 4. Cross-Facility Comparative Analysis
Compare the 4 manufacturing centers to identify which sites demonstrate proactive compliance vs high regulatory risk.


In [ ]:
factory_analysis = df_eq.groupby('Factory').agg(
    total_roles=('Equality Score', 'count'),
    mean_score=('Equality Score', 'mean'),
    median_score=('Equality Score', 'median'),
    worst_score=('Equality Score', 'min'),
    best_score=('Equality Score', 'max')
).reset_index().sort_values(by='mean_score')

# Cross-tabulation of risk tiers by facility
cross_tab = pd.crosstab(df_eq['Factory'], df_eq['Equality class'], margins=True)
print("Risk Category Matrix by Facility:")
print(cross_tab)
print("\nFacility Equality Averages:")
print(factory_analysis)


## 5. Visualizing Equity Stratification
Generate diagnostic charts showing seniority progression vs pay balance.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Mean Score by Job Seniority
role_means = df_eq.groupby('Job Role')['Equality Score'].mean().reindex(role_order)
colors_role = ['#d73027' if v < -20 else ('#fc8d59' if v < -10 else '#91cf60') for v in role_means.values]

axes[0].barh(role_order[::-1], role_means.values[::-1], color=colors_role[::-1], edgecolor='black', alpha=0.85)
axes[0].axvline(0, color='black', linestyle='-', linewidth=1)
axes[0].axvline(-10, color='orange', linestyle='--', label='Fair (-10)')
axes[0].axvline(-20, color='red', linestyle='--', label='Discriminative (-20)')
axes[0].set_title('Mean Equality Score by Job Seniority', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Score (0 = Balance, Negative = Male Advantage)')
axes[0].legend(loc='lower left')

# Plot 2: Facility Risk Category Composition
ct_plot = pd.crosstab(df_eq['Factory'], df_eq['Equality class'])[['Fair', 'Unfair', 'Highly Discriminative']]
ct_plot.plot(kind='bar', stacked=True, color=['#2ecc71', '#e67e22', '#e74c3c'], ax=axes[1], edgecolor='black', alpha=0.85)
axes[1].set_title('Risk Tier Breakdown by Manufacturing Hub', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Role Count')
axes[1].set_xlabel('Facility')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Equality Class', loc='upper right')

plt.tight_layout()
plt.show()


## 6. Export Completed Audit Table
Write audited results to `../data/processed/equality_classified.xlsx`.


In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df_eq.to_excel('../data/processed/equality_classified.xlsx', index=False)
print("Export complete: ../data/processed/equality_classified.xlsx")
